# धडा ०४ - टूल वापर डिझाईन पॅटर्न

या धड्यात तुम्हाला मायक्रोसॉफ्ट एजंट फ्रेमवर्क (पायथन) वापरून AI एजंटसाठी **टूल वापर** डिझाईन पॅटर्न शिकवला जाईल. आम्ही खालील गोष्टींचा समावेश करतो:

- `@tool` डेकोरेटर आणि टाइप केलेल्या पॅरामीटर्ससह फंक्शन टूल्सची व्याख्या करणे
- मॉडलला प्रत्येक टूल काय करते हे कळावे म्हणून टूल स्कीमा पुरवणे
- `approval_mode` सह टूलची अंमलबजावणी नियंत्रणात ठेवणे
- Pydantic मॉडेल्स आणि `response_format` द्वारे **रचनेत आउटपुट** परत करणे

परस्थिती अशी आहे की एक **ट्रॅव्हल बुकिंग एजंट** ज्याला गंतव्यस्थान तपासता येते, उपलब्धता तपासता येते, आणि फ्लाइट माहिती मिळवता येते.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool डेकोरेटरसह साधने परिभाषित करणे

`@tool` डेकोरेटर एक साधा Python फंक्शन साधनात रूपांतरित करतो जे एजंट कॉल करू शकतो.
महत्त्वाचे मुद्दे:

- **docstring** ही साधनाचे वर्णन बनते ज्याला मॉडेल बघते.
- **टाइप अ‍ॅनोटेशन्स** (वर्णनांसह `Annotated` सह) साधनाच्या स्कीमाला परिभाषित करतात.
- `approval_mode` नियंत्रित करते की वापरकर्त्याने प्रत्येक कॉलला कार्यान्वित होण्यापूर्वी मंजुरी द्यावी का.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## अनेक टूल्ससह एजंट तयार करणे

वापरकर्त्याच्या प्रश्नाचे उत्तर देण्यासाठी मॉडेलला आवश्यक असलेल्या कोणत्याही टूलचा वापर करू शकणार्या सर्व तीन टूल्स क्लायंटला द्या.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## साधनांसह संरचित आउटपुट

`response_format` एका Pydantic मॉडेलवर सेट केल्याने, एजंटला मुक्त स्वरूपातील मजकूराऐवजी एक सुसंगत प्रकाराचा JSON ऑब्जेक्ट परत करावा लागतो. जेव्हा खालील कोड परिणाम प्रोग्रामॅटिक प्रकारे वापरायला हवा असतो तेव्हा हे उपयुक्त ठरते.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## साधन मंजुरी नमुने

`@tool` वरील `approval_mode` पॅरामीटर नियंत्रित करते की एक साधन कॉल कार्यान्वित होण्याआधी मानवी मंजुरी आवश्यक आहे का:

| मोड | वर्तन |
|---|---|
| `"never_require"` | साधन आपोआप चालते — वापरकर्त्याची पुष्टी आवश्यक नाही. |
| `"always_require"` | प्रत्येक कॉल कार्यान्वित होण्याआधी वापरकर्त्याद्वारे मंजूर करणे आवश्यक आहे. |

असे साधन जे साइड-इफेक्ट्स करतात (उदा. फ्लाइट बुक करणे, क्रेडिट कार्ड आकारणी करणे) त्यांच्यासाठी `"always_require"` वापरा जेणेकरून मानवी सहभाग कायम राहील.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

या धड्यात तुम्ही शिकले कसे:

1. **टूल्स परिभाषित करा** `@tool` डेकोरेटर वापरून टाइप केलेले पॅरामीटर्स आणि डॉकस्ट्रिंग्जसह जे टूल स्कीमा म्हणून काम करतात.
2. **अनेक टूल्स एकत्र करा** जेणेकरून एजंट त्यांना अनुक्रमे कॉल करून जटिल प्रश्नांची उत्तरे देऊ शकेल.
3. **संरचित आउटपुट परत करा** एक Pydantic मॉडेल `response_format` म्हणून पास करून.
4. **टूल मंजुरी नियंत्रण करा** `approval_mode` वापरून संवेदनशील क्रियांसाठी मानवी नियंत्रण ठेवण्यासाठी.

हे नमुने भरोसेमंद, उत्पादनासाठी तयार एजंट तयार करण्यासाठी पाया तयार करतात जे सुरक्षितपणे बाह्य प्रणालींशी संवाद साधू शकतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
